# 01: Load and Profile SAML-D
Load the raw SAML-D transaction file into DuckDB and profile its structure, volumes, and label distribution.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

RAW_CSV = Path("../data/raw/SAML-D.csv")
DB_PATH = Path("../data/processed/aml.duckdb")

con = duckdb.connect(str(DB_PATH))
print("Connected to:", DB_PATH.resolve())

Connected to: C:\Users\jhunt\Projects\aml-tm-scenario-tuning\data\processed\aml.duckdb


In [2]:
con.execute(f"""
    CREATE OR REPLACE TABLE transactions_raw AS
    SELECT *
    FROM read_csv_auto('{RAW_CSV.as_posix()}')
""")

con.sql("SELECT COUNT(*) AS n_rows FROM transactions_raw").df()

,n_rows
0,9504852


In [3]:
con.sql("DESCRIBE transactions_raw").df()

,column_name,column_type,null,key,default,extra
0,Time,TIME,YES,None,None,None
1,Date,DATE,YES,None,None,None
2,Sender_account,BIGINT,YES,None,None,None
3,Receiver_account,BIGINT,YES,None,None,None
4,Amount,DOUBLE,YES,None,None,None
5,Payment_currency,VARCHAR,YES,None,None,None
6,Received_currency,VARCHAR,YES,None,None,None
7,Sender_bank_location,VARCHAR,YES,None,None,None
8,Receiver_bank_location,VARCHAR,YES,None,None,None
9,Payment_type,VARCHAR,YES,None,None,None


In [4]:
con.sql("SELECT * FROM transactions_raw LIMIT 10").df()

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
5,10:35:21,2022-10-07,8974559268,3143547511,5130.99,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Group
6,10:35:23,2022-10-07,980191499,8577635959,12176.52,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Small_Fan_Out
7,10:35:23,2022-10-07,8057793308,9350896213,56.90,UK pounds,UK pounds,UK,UK,Credit card,0,Normal_Small_Fan_Out
8,10:35:26,2022-10-07,6116657264,656192169,4738.45,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Fan_Out
9,10:35:29,2022-10-07,7421451752,2755709071,5883.87,Indian rupee,UK pounds,UK,UK,Credit card,0,Normal_Fan_Out


In [5]:
con.sql("""
    SELECT
        MIN(Date)                          AS first_date,
        MAX(Date)                          AS last_date,
        COUNT(*)                           AS n_txns,
        SUM(Is_laundering)                 AS n_laundering,
        ROUND(AVG(Is_laundering) * 100, 4) AS pct_laundering
    FROM transactions_raw
""").df()

,first_date,last_date,n_txns,n_laundering,pct_laundering
0,2022-10-07,2023-08-23,9504852,9873.0,0.1039


In [6]:
con.sql("""
    SELECT
        Payment_type,
        COUNT(*)                           AS n_txns,
        SUM(Is_laundering)                 AS n_laundering,
        ROUND(AVG(Is_laundering) * 100, 3) AS pct_laundering,
        ROUND(AVG(Amount), 2)              AS avg_amount
    FROM transactions_raw
    GROUP BY Payment_type
    ORDER BY n_txns DESC
""").df()

,Payment_type,n_txns,n_laundering,pct_laundering,avg_amount
0,Credit card,2012909,1136.0,0.056,9095.75
1,Debit card,2012103,1124.0,0.056,9130.91
2,Cheque,2011419,1087.0,0.054,9112.41
3,ACH,2008807,1159.0,0.058,9095.97
4,Cross-border,933931,2628.0,0.281,10146.99
5,Cash Withdrawal,300477,1334.0,0.444,153.48
6,Cash Deposit,225206,1405.0,0.624,2157.18


In [7]:
con.sql("""
    SELECT
        Is_laundering,
        Laundering_type,
        COUNT(*) AS n_txns
    FROM transactions_raw
    GROUP BY Is_laundering, Laundering_type
    ORDER BY Is_laundering DESC, n_txns DESC
""").df()

,Is_laundering,Laundering_type,n_txns
0,1,Structuring,1870
1,1,Cash_Withdrawal,1334
2,1,Deposit-Send,945
3,1,Smurfing,932
4,1,Layered_Fan_In,656
5,1,Layered_Fan_Out,529
6,1,Stacked Bipartite,506
7,1,Behavioural_Change_1,394
8,1,Bipartite,383
9,1,Cycle,382


## Key Findings: Initial Profile

- **Scope and base rate.** The dataset contains 9,504,852 transactions from October 7, 2022 through August 23, 2023, of which 9,873 are labeled as laundering. That is a base rate of roughly 0.10%, or about 1 in every 960 transactions. This baseline matters because any scenario or model has to be judged against it, and a precision near 0.10% would be no better than random selection.

- **Risk is concentrated in cash and cross-border activity.** Cash deposits, cash withdrawals, and cross-border payments make up only about 15% of total transaction volume, yet they account for roughly 54% of all laundering activity. Cash deposits carry the highest laundering rate at 0.62%, which is about 11 times higher than card, cheque, and ACH payments at roughly 0.06%. This supports a risk-based approach where monitoring coverage and scenario design are prioritized toward these channels.

- **Structuring is the most common typology.** Of the 17 laundering typologies in the data, structuring is the largest with 1,870 transactions, or about 19% of all laundering activity. This makes it a strong candidate for the primary scenario, where I can compare a rules-based approach against a machine learning model on the same typology.

- **Normal activity closely resembles suspicious patterns.** The 11 normal typologies include behaviors such as fan-in, fan-out, and group payments, which mirror several of the laundering typologies. For example, normal small fan-out activity alone accounts for nearly 3.5 million transactions. This overlap is a key driver of false positives in rules-based monitoring and is the reason segmentation and threshold tuning will be central to this project.

- **Data considerations.** There is no customer or KYC table, so each account serves as a proxy for a customer, which limits segmentation to behavioral attributes rather than risk ratings or customer type. Date and time are stored separately and will need to be combined for time-window scenarios. The source data also contains a misspelled typology label ("Normal_Foward"), which I am documenting rather than correcting in order to preserve data lineage.